In [1]:
!pip install -U "langchain>=1.3.14"

   ---------------------------------------- 0.0/570.0 kB ? eta -:--:--
   ---------------------------------------- 570.0/570.0 kB 9.9 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.5.4
    Uninstalling langchain-core-1.5.4:
      Successfully uninstalled langchain-core-1.5.4
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.15
    Uninstalling langchain-1.3.15:
      Successfully uninstalled langchain-1.3.15


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.0.19 requires langchain-core<0.2,>=0.1.21, but you have langchain-core 1.6.0 which is incompatible.
langchain-community 0.0.19 requires langsmith<0.1,>=0.0.83, but you have langsmith 0.10.10 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install  langchain-openai langchain-community langgraph python-dotenv langchain-mcp-adapters langchain-chroma chromadb pypdf

INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
Using cached langchain_community-0.4.2-py3-none-any.whl (2.4 MB)
  Attempting uninstall: langchain-community
    Found existing installation: langchain-community 0.0.19
    Uninstalling langchain-community-0.0.19:
      Successfully uninstalled langchain-community-0.0.19



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
from getpass import getpass

if not os.environ.get("OPENROUTER_API_KEY"):
    api_key = os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")
    print(f"OpenRouter API key set in environment variable OPENROUTER_API_KEY: {api_key[0:4]}")

OpenRouter API key set in environment variable OPENROUTER_API_KEY: sk-o


In [4]:
from langchain_openai import ChatOpenAI

# OpenRouter exposes an OpenAI-compatible API, so ChatOpenAI works with a custom base_url
model = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0,
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)
model

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.16', 'langchain-openai': '1.4.2'}}, client=<openai.resources.chat.completions.completions.Completions object at 0x00000178487D2ED0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000017848A4B500>, root_client=<openai.OpenAI object at 0x0000017846E0BD40>, root_async_client=<openai.AsyncOpenAI object at 0x000001784878A960>, model_name='openai/gpt-4o-mini', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://openrouter.ai/api/v1', stream_chunk_timeout=120.0)

In [5]:
free_selector_model = ChatOpenAI(
    model="openrouter/free",
    temperature=0,
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

free_selector_model

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.16', 'langchain-openai': '1.4.2'}}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000017844CD8590>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000017848AF0F80>, root_client=<openai.OpenAI object at 0x0000017848A21E80>, root_async_client=<openai.AsyncOpenAI object at 0x0000017844CD8530>, model_name='openrouter/free', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://openrouter.ai/api/v1', stream_chunk_timeout=120.0)

In [6]:
# --- Core LangChain ---
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.tools import tool as tool_rt, ToolRuntime

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

from langchain.agents.middleware import (
    SummarizationMiddleware,
    HumanInTheLoopMiddleware,
    ModelCallLimitMiddleware,
    ToolCallLimitMiddleware,
    ModelFallbackMiddleware,
    PIIMiddleware,
    TodoListMiddleware,
    LLMToolSelectorMiddleware,
    ToolErrorMiddleware,
    ToolRetryMiddleware,
    ModelRetryMiddleware,
    LLMToolEmulator,
    ContextEditingMiddleware,
    ClearToolUsesEdit,
)

### Cinebot Agent

In [7]:
@tool
def check_showtime(movie_title:str)-> str:
    """
    Check if a movie is currently showing in theaters.
    """
    fake_showtimes = {
        "interstellar": "7:00 PM and 10:15 PM",
        "dune part two": "9:30 PM only",
        "oppenheimer": "Sold out for tonight",
    }
    return fake_showtimes.get(movie_title.lower(), "No showtimes available for this movie.")

In [8]:
@tool
def book_seats(movie_title:str,seat_count:int)-> str:
    """
    Book seats for a movie.
    """
    return f"Successfully booked {seat_count} seats for '{movie_title}'. Enjoy the show!"

In [9]:
@tool
def cancel_booking(booking_id: str) -> str:
    """Cancel an existing booking. Irreversible."""
    return f"Booking {booking_id} cancelled."

In [10]:
@tool
def check_order_status(booking_id: str) -> str:
    """Check the status of an existing booking."""
    return f"Booking {booking_id}: confirmed, 2 seats, Interstellar, 7:00 PM."

In [11]:
@tool
def get_refund_policy() -> str:
    """Get the cinema's refund policy -- exact wording, not to be paraphrased."""
    return "Refunds available up to 2 hours before showtime. No refunds after that."

In [12]:
@tool
def lookup_seat_map(movie_title: str, seat_number: str) -> str:
    """Look up a specific seat -- fails if the seat number format is wrong."""
    if not seat_number or not seat_number[0].isalpha():
        raise ValueError(f"Malformed seat number '{seat_number}' -- expected a letter+number like 'A12'.")
    return f"Seat {seat_number} for {movie_title}: available."

In [13]:
cinebot_tools = [
    check_showtime,
    book_seats,
    cancel_booking,
    check_order_status,
    get_refund_policy,
    lookup_seat_map,
]

In [14]:
def pretty_print_messages(result):
    for i, message in enumerate(result.get("messages", []), 1):
        print(f"\n{'=' * 80}")
        print(f"Message {i}: {message.__class__.__name__}")
        print("=" * 80)

        # Basic message information
        print(f"ID: {getattr(message, 'id', None)}")

        # Message content
        content = getattr(message, "content", "")
        if content:
            print("\nContent:")
            print(content)

        # Tool calls
        tool_calls = getattr(message, "tool_calls", None)
        if tool_calls:
            print("\nTool Calls:")
            for tool in tool_calls:
                print(f"  • {tool['name']}")
                print(f"    Args: {tool['args']}")
                print(f"    ID:   {tool['id']}")

        # Tool message information
        tool_call_id = getattr(message, "tool_call_id", None)
        if tool_call_id:
            print(f"\nTool Call ID: {tool_call_id}")

        # Summarization information
        additional_kwargs = getattr(message, "additional_kwargs", {})
        if additional_kwargs.get("lc_source"):
            print(f"\nSource: {additional_kwargs['lc_source']}")

    print(f"\n{'=' * 80}")
    print("END OF MESSAGE HISTORY")
    print("=" * 80)


#### Summarization Middleware

In [15]:
summarization_agent = create_agent(
    tools=cinebot_tools,
    model = model,
    middleware=[
        SummarizationMiddleware(
            model = model,
            trigger = ("tokens", 200),
            keep = ("messages", 1),),
    ]
    
)

In [16]:
print(summarization_agent.invoke({"messages":[("user","Hi I am Vishal")]}))

{'messages': [HumanMessage(content='Hi I am Vishal', additional_kwargs={}, response_metadata={}, id='23532256-4c27-4c0e-905d-8faa10a121bb'), AIMessage(content='Hello Vishal! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 193, 'total_tokens': 205, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'video_tokens': 0}, 'cost': 3.615e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 3.615e-05, 'upstream_inference_prompt_cost': 2.895e-05, 'upstream_inference_completions_cost': 7.2e-06}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_f282d56213', 'id': 'gen-1787428379-cABsqAEsGvKScPoPislN', 'finish_reason': 'stop', 'logprobs': None}, id=

In [17]:
print(summarization_agent.invoke({"messages":[("user","Who am I?")]}))

{'messages': [HumanMessage(content='Who am I?', additional_kwargs={}, response_metadata={}, id='d43a2a69-5da9-484c-8e34-0d80407047e9'), AIMessage(content="I don't have access to personal data about individuals unless it has been shared with me in the course of our conversation. Therefore, I can't tell you who you are. However, I can help answer questions or provide information on a wide range of topics! How can I assist you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 192, 'total_tokens': 250, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'video_tokens': 0}, 'cost': 6.36e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 6.36e-05, 'upstream_inference_prompt_cost': 2.88e-05, 'upstream_infer

In [18]:
result = summarization_agent.invoke({"messages": [("user", "Is Interstellar showing tonight? also please make sure that you book me a ticket, refund me if it is not available,also share the refund policy for me to go through, also check my order status for book_1234")]})


pretty_print_messages(result)


Message 1: HumanMessage
ID: f2e00f92-b7c5-4ea7-bbb9-ef72a890c27d

Content:
Here is a summary of the conversation to date:

## SESSION INTENT

The user wants to check if the movie "Interstellar" is showing tonight, book a ticket for it, and inquire about the refund policy. Additionally, they want to check the order status for a specific booking (book_1234).

## SUMMARY

The user requested information about the availability of "Interstellar" for tonight and asked for a ticket to be booked. They also requested a refund if the ticket is not available and asked for the refund policy to review. Furthermore, the user inquired about the status of their order with the identifier book_1234.

## ARTIFACTS

None

## NEXT STEPS

1. Check if "Interstellar" is showing tonight.
2. Book a ticket for the user if available.
3. Provide the refund policy for the user to review.
4. Check the order status for book_1234.

Source: summarization

Message 2: AIMessage
ID: lc_run--01a02b09-2aa1-7aa1-befa-a3ceff0

#### HITL (Human in the Loop)

In [19]:
guarded_agent = create_agent(
    model=model,
    tools=cinebot_tools,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"cancel_booking": {"allowed_decisions": ["approve", "edit", "reject", "respond"]}}
        ),
    ],
    checkpointer=InMemorySaver(),  # REQUIRED -- HITL needs to pause and later resume
)

config = {'configurable':{'thread_id':'hitl-demo-live'}}

In [20]:
print(guarded_agent.invoke({"messages":[("user","Hi I am Vishal")]}),config=config)

ValueError: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id

In [ ]:
print(summarization_agent.invoke({"messages": [("user", "Who am I ? ")]}))

In [21]:
result = guarded_agent.invoke({"messages": [("user", "Please cancel booking BK1042")]}, config=config)
from rich import print

print(result)

{
    'messages': [
        HumanMessage(
            content='Please cancel booking BK1042',
            additional_kwargs={},
            response_metadata={},
            id='b943a4bc-d129-4dca-95ec-3540105b7d23'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 17,
                    'prompt_tokens': 194,
                    'total_tokens': 211,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': 0,
                        'cached_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 3.93e-05,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 3.93e-05,
                        'upstream_inference_prompt_cost': 2.91e-05,
                        'upstream_inference_completions_cost': 1.02e-05
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-4o-mini',
                'system_fingerprint': 'fp_f282d56213',
                'id': 'gen-1787428491-idoz1HCxzZ2UJSmcO9Zr',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a02b0a-505a-7e51-8962-247676329a37-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'booking_id': 'BK1042'},
                    'id': 'call_LLESQklMqA3r4wefv9k7uWjc',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 194,
                'output_tokens': 17,
                'total_tokens': 211,
                'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        )
    ],
    '__interrupt__': [
        Interrupt(
            value={
                'action_requests': [
                    {
                        'name': 'cancel_booking',
                        'args': {'booking_id': 'BK1042'},
                        'description': "Tool execution requires approval\n\nTool: cancel_booking\nArgs: 
{'booking_id': 'BK1042'}"
                    }
                ],
                'review_configs': [
                    {
                        'action_name': 'cancel_booking',
                        'allowed_decisions': ['approve', 'edit', 'reject', 'respond']
                    }
                ]
            },
            id='287c9fdb578d96394f53583b258f7107'
        )
    ]
}

In [22]:
state = guarded_agent.get_state(config)

In [23]:
resumed_result = guarded_agent.invoke(Command(resume={"decisions":[{"type":"approve"}]}),config=config)
print(resumed_result)

{
    'messages': [
        HumanMessage(
            content='Please cancel booking BK1042',
            additional_kwargs={},
            response_metadata={},
            id='b943a4bc-d129-4dca-95ec-3540105b7d23'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 17,
                    'prompt_tokens': 194,
                    'total_tokens': 211,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': 0,
                        'cached_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 3.93e-05,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 3.93e-05,
                        'upstream_inference_prompt_cost': 2.91e-05,
                        'upstream_inference_completions_cost': 1.02e-05
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-4o-mini',
                'system_fingerprint': 'fp_f282d56213',
                'id': 'gen-1787428491-idoz1HCxzZ2UJSmcO9Zr',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a02b0a-505a-7e51-8962-247676329a37-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'booking_id': 'BK1042'},
                    'id': 'call_LLESQklMqA3r4wefv9k7uWjc',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 194,
                'output_tokens': 17,
                'total_tokens': 211,
                'input_token_details': {'audio': 0, 'cache_creation': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='Booking BK1042 cancelled.',
            name='cancel_booking',
            id='6e10d57e-9673-45f3-8814-0c3061e4e87f',
            tool_call_id='call_LLESQklMqA3r4wefv9k7uWjc'
        ),
        AIMessage(
            content='Your booking with ID BK1042 has been successfully cancelled.',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 13,
                    'prompt_tokens': 225,
                    'total_tokens': 238,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': 0,
                        'cached_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 4.155e-05,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 4.155e-05,
                        'upstream_inference_prompt_cost': 3.375e-05,
                        'upstream_inference_completions_cost': 7.8e-06
                    }
                },
                'model_provider': 'openai',
  

In [ ]:
state.next

In [ ]:
def run_interactive_hitl_demo(agent, config):
    """A genuinely interactive HITL loop -- ask out loud, type the answer, watch it apply live."""
    state = agent.get_state(config)
    if not state.next:
        print("Nothing is currently paused for approval.")
        return

    print("The agent wants to call a guarded tool. Choose a decision:")
    print("  1) approve  -- run it exactly as proposed")
    print("  2) edit     -- run it, but change the booking_id first")
    print("  3) reject   -- block it, with a reason sent back to the agent")
    print("  4) respond  -- answer a question instead of deciding on the action")

    choice = input("Type 1, 2, 3, or 4: ").strip()

    if choice == "1":
        decision = {"type": "approve"}
    elif choice == "2":
        new_id = input("New booking_id to use instead: ").strip()
        decision = {"type": "edit", "args": {"booking_id": new_id}}
    elif choice == "3":
        reason = input("Reason for rejecting: ").strip()
        decision = {"type": "reject", "message": reason}
    elif choice == "4":
        answer = input("Your response to the agent: ").strip()
        decision = {"type": "respond", "message": answer}
    else:
        print("Not a valid choice -- try again.")
        return

    resumed = agent.invoke(Command(resume={"decisions": [decision]}), config=config)
    print()
    print("Agent's final response:", resumed["messages"][-1].content)

In [ ]:
config = {'configurable':{'thread_id':'hitl-demo-live-2'}}

guarded_agent = create_agent(
    model=model,
    tools=cinebot_tools,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"cancel_booking": {"allowed_decisions": ["approve", "edit", "reject", "respond"]}}
        ),
    ],
    checkpointer=InMemorySaver(),  # REQUIRED -- HITL needs to pause and later resume
)


result = guarded_agent.invoke({"messages": [("user", "Please cancel booking BK1042")]}, config=config)


In [ ]:
run_interactive_hitl_demo(guarded_agent, config)

#### Model Call Limit

In [27]:
model_call_limit_agent = create_agent(
    model=model,
    tools=cinebot_tools,
    middleware=[
        ModelCallLimitMiddleware(
            thread_limit=2,
            run_limit=1,
            exit_behavior="end",
        ),
    ],
)

In [25]:

result = model_call_limit_agent.invoke(
    {"messages": [("user", "Can you tell me cinema's refund policy? ")]},
    config={"configurable": {"thread_id": "call-limit-demo-4"}},
)

In [26]:
print(result)

{
    'messages': [
        HumanMessage(
            content="Can you tell me cinema's refund policy? ",
            additional_kwargs={},
            response_metadata={},
            id='6a161a4b-6f66-4373-ba02-dcc0f0550107'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 12,
                    'prompt_tokens': 198,
                    'total_tokens': 210,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': 0,
                        'cached_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 3.69e-05,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 3.69e-05,
                        'upstream_inference_prompt_cost': 2.97e-05,
                        'upstream_inference_completions_cost': 7.2e-06
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-4o-mini',
                'system_fingerprint': 'fp_f282d56213',
                'id': 'gen-1787428586-NnJi72ViRHojzdy78xQG',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a02b0b-c6c5-7540-a504-6f2de9266da7-0',
            tool_calls=[
                {
                    'name': 'get_refund_policy',
                    'args': {},
                    'id': 'call_arHdHg5c9mEYGG4gJ0yhKReZ',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 198,
                'output_tokens': 12,
                'total_tokens': 210,
                'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='Refunds available up to 2 hours before showtime. No refunds after that.',
            name='get_refund_policy',
            id='4db4bfa7-83df-4527-90d1-d1ab3f6319cd',
            tool_call_id='call_arHdHg5c9mEYGG4gJ0yhKReZ'
        ),
        AIMessage(
            content="The cinema's refund policy states that refunds are available up to 2 hours before showtime. No
refunds are allowed after that.",
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 27,
                    'prompt_tokens': 237,
                    'total_tokens': 264,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': 0,
                        'cached_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 5.175e-05,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 5.175e-05,
                        'upstream_inference_prompt_cost': 3.555e-05,
                        'upstream_inference_complet

In [28]:
result_invoke_2= model_call_limit_agent.invoke(
    {"messages": [("user", "cancel my booking B123? ")]},
    config={"configurable": {"thread_id": "call-limit-demo-4"}},
)
print(result_invoke_2)

{
    'messages': [
        HumanMessage(
            content='cancel my booking B123? ',
            additional_kwargs={},
            response_metadata={},
            id='542df86e-7e1d-4409-95fe-4bc422d3fa4b'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 16,
                    'prompt_tokens': 195,
                    'total_tokens': 211,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': 0,
                        'cached_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 3.885e-05,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 3.885e-05,
                        'upstream_inference_prompt_cost': 2.925e-05,
                        'upstream_inference_completions_cost': 9.6e-06
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-4o-mini',
                'system_fingerprint': 'fp_f282d56213',
                'id': 'gen-1787428667-NznmuKPmZ8rH4FwqgkEo',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a02b0c-ff9c-7863-b504-ec3883aa6582-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'booking_id': 'B123'},
                    'id': 'call_WivAfPKBIbDtHd6XPIsyDlK2',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 195,
                'output_tokens': 16,
                'total_tokens': 211,
                'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='Booking B123 cancelled.',
            name='cancel_booking',
            id='451ae51a-9fc6-4108-81dd-f2b6d2752175',
            tool_call_id='call_WivAfPKBIbDtHd6XPIsyDlK2'
        ),
        AIMessage(
            content='Model call limits exceeded: run limit (1/1)',
            additional_kwargs={},
            response_metadata={},
            id='2ab76f05-7f19-4e7e-a9fc-8dbbbb20f17d',
            tool_calls=[],
            invalid_tool_calls=[]
        )
    ]
}

In [29]:
result_invoke_3= model_call_limit_agent.invoke(
    {"messages": [("user", "cancel my booking B123? ")]},
    config={"configurable": {"thread_id": "call-limit-demo-4"}},
)
print(result_invoke_3)

{
    'messages': [
        HumanMessage(
            content='cancel my booking B123? ',
            additional_kwargs={},
            response_metadata={},
            id='d7fc4d5f-8b95-45b4-be9d-6ae3b8634170'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 16,
                    'prompt_tokens': 195,
                    'total_tokens': 211,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': 0,
                        'cached_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 3.885e-05,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 3.885e-05,
                        'upstream_inference_prompt_cost': 2.925e-05,
                        'upstream_inference_completions_cost': 9.6e-06
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-4o-mini',
                'system_fingerprint': 'fp_f282d56213',
                'id': 'gen-1787428685-79SHB42C70sYNTVn6rpy',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a02b0d-4856-7860-bdd6-b55c4ec24623-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'booking_id': 'B123'},
                    'id': 'call_vnGiywcsE32Qu1gj8SO5pFun',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 195,
                'output_tokens': 16,
                'total_tokens': 211,
                'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='Booking B123 cancelled.',
            name='cancel_booking',
            id='454e4179-dd32-4050-a200-1058ba7f131c',
            tool_call_id='call_vnGiywcsE32Qu1gj8SO5pFun'
        ),
        AIMessage(
            content='Model call limits exceeded: run limit (1/1)',
            additional_kwargs={},
            response_metadata={},
            id='8742c364-eedb-471a-b5ce-20013449eb39',
            tool_calls=[],
            invalid_tool_calls=[]
        )
    ]
}

#### Model Fallback

In [30]:
model_fallback_agent = create_agent(
    model=model,  # Primary model: OpenAI GPT-4o Mini through OpenRouter
    tools=cinebot_tools,
    middleware=[
        ModelFallbackMiddleware(
            free_selector_model,  # Fallback: Gemini through OpenRouter
        )
    ],
)

In [31]:
result = model_fallback_agent.invoke( {"messages": [("user", "Summarize my chat? ")]},)
print(result)

{
    'messages': [
        HumanMessage(
            content='Summarize my chat? ',
            additional_kwargs={},
            response_metadata={},
            id='10d7a077-bed4-4936-85cd-c399b13bfb7d'
        ),
        AIMessage(
            content="You haven't had a chat yet. Please let me know what you'd like to discuss or ask!",
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 20,
                    'prompt_tokens': 195,
                    'total_tokens': 215,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': 0,
                        'cached_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 4.125e-05,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 4.125e-05,
                        'upstream_inference_prompt_cost': 2.925e-05,
                        'upstream_inference_completions_cost': 1.2e-05
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-4o-mini',
                'system_fingerprint': 'fp_f282d56213',
                'id': 'gen-1787428729-oy0HfUTHdmfjIuBrualm',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a02b0d-f2ae-7710-8a0e-baedfd4582a1-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 195,
                'output_tokens': 20,
                'total_tokens': 215,
                'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        )
    ]
}

#### Tool Call Limit

In [34]:

tool_limited_agent = create_agent(
    model=model,
    tools=cinebot_tools,
    checkpointer=InMemorySaver(),
    middleware=[
        ToolCallLimitMiddleware(run_limit=8),                              # global, this turn
        ToolCallLimitMiddleware(tool_name="cancel_booking", thread_limit=2, run_limit=1),  # tighter, one tool, whole conversation
    ],
)

In [35]:
config = {'configurable': {'thread_id': 'tool-limit-demo'}}

for i in range(3):
  result = tool_limited_agent.invoke({"messages": [("user", f"Please cancel my Booking with ID B{100+i} ? ")]}, config=config)
  print(result)

    

{
    'messages': [
        HumanMessage(
            content='Please cancel my Booking with ID B100 ? ',
            additional_kwargs={},
            response_metadata={},
            id='876492df-63db-4123-a797-6eb59566ecbb'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 16,
                    'prompt_tokens': 198,
                    'total_tokens': 214,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': 0,
                        'cached_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 3.93e-05,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 3.93e-05,
                        'upstream_inference_prompt_cost': 2.97e-05,
                        'upstream_inference_completions_cost': 9.6e-06
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-4o-mini',
                'system_fingerprint': 'fp_f282d56213',
                'id': 'gen-1787428847-QPK6gdbiggMfCVxAsVtO',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a02b0f-c209-7772-9d1c-9ecb13ba4a15-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'booking_id': 'B100'},
                    'id': 'call_igcUDlFbQEYzrzwp7n8ZEYLo',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 198,
                'output_tokens': 16,
                'total_tokens': 214,
                'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='Booking B100 cancelled.',
            name='cancel_booking',
            id='ffef2ed2-0887-4173-b13d-36ef442ce89a',
            tool_call_id='call_igcUDlFbQEYzrzwp7n8ZEYLo'
        ),
        AIMessage(
            content='Your booking with ID B100 has been successfully cancelled.',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 12,
                    'prompt_tokens': 227,
                    'total_tokens': 239,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': 0,
                        'cached_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 4.125e-05,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 4.125e-05,
                        'upstream_inference_prompt_cost': 3.405e-05,
                        'upstream_inference_completions_cost': 7.2e-06
                    }
                },
                'model_provider': 'openai

{
    'messages': [
        HumanMessage(
            content='Please cancel my Booking with ID B100 ? ',
            additional_kwargs={},
            response_metadata={},
            id='876492df-63db-4123-a797-6eb59566ecbb'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 16,
                    'prompt_tokens': 198,
                    'total_tokens': 214,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': 0,
                        'cached_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 3.93e-05,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 3.93e-05,
                        'upstream_inference_prompt_cost': 2.97e-05,
                        'upstream_inference_completions_cost': 9.6e-06
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-4o-mini',
                'system_fingerprint': 'fp_f282d56213',
                'id': 'gen-1787428847-QPK6gdbiggMfCVxAsVtO',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a02b0f-c209-7772-9d1c-9ecb13ba4a15-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'booking_id': 'B100'},
                    'id': 'call_igcUDlFbQEYzrzwp7n8ZEYLo',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 198,
                'output_tokens': 16,
                'total_tokens': 214,
                'input_token_details': {'audio': 0, 'cache_creation': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='Booking B100 cancelled.',
            name='cancel_booking',
            id='ffef2ed2-0887-4173-b13d-36ef442ce89a',
            tool_call_id='call_igcUDlFbQEYzrzwp7n8ZEYLo'
        ),
        AIMessage(
            content='Your booking with ID B100 has been successfully cancelled.',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 12,
                    'prompt_tokens': 227,
                    'total_tokens': 239,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': 0,
                        'cached_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 4.125e-05,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 4.125e-05,
                        'upstream_inference_prompt_cost': 3.405e-05,
                        'upstream_inference_completions_cost': 7.2e-06
                    }
                },
                'model_provider': 'openai

{
    'messages': [
        HumanMessage(
            content='Please cancel my Booking with ID B100 ? ',
            additional_kwargs={},
            response_metadata={},
            id='876492df-63db-4123-a797-6eb59566ecbb'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 16,
                    'prompt_tokens': 198,
                    'total_tokens': 214,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': 0,
                        'cached_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 3.93e-05,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 3.93e-05,
                        'upstream_inference_prompt_cost': 2.97e-05,
                        'upstream_inference_completions_cost': 9.6e-06
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-4o-mini',
                'system_fingerprint': 'fp_f282d56213',
                'id': 'gen-1787428847-QPK6gdbiggMfCVxAsVtO',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a02b0f-c209-7772-9d1c-9ecb13ba4a15-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'booking_id': 'B100'},
                    'id': 'call_igcUDlFbQEYzrzwp7n8ZEYLo',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 198,
                'output_tokens': 16,
                'total_tokens': 214,
                'input_token_details': {'audio': 0, 'cache_creation': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='Booking B100 cancelled.',
            name='cancel_booking',
            id='ffef2ed2-0887-4173-b13d-36ef442ce89a',
            tool_call_id='call_igcUDlFbQEYzrzwp7n8ZEYLo'
        ),
        AIMessage(
            content='Your booking with ID B100 has been successfully cancelled.',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 12,
                    'prompt_tokens': 227,
                    'total_tokens': 239,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': 0,
                        'cached_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 4.125e-05,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 4.125e-05,
                        'upstream_inference_prompt_cost': 3.405e-05,
                        'upstream_inference_completions_cost': 7.2e-06
                    }
                },
                'model_provider': 'openai

#### PII Detection

In [36]:
pii_agent = create_agent(
    model=model,
    tools=cinebot_tools,
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
    ],
)

In [37]:
result = pii_agent.invoke({
    "messages": [("user", "My email is priya@example.com and my credit card is 4111-1111-1111-1234, can you check showtimes for Dune?")]
})

print(result)

{
    'messages': [
        HumanMessage(
            content='My email is [REDACTED_EMAIL] and my credit card is 4111-1111-1111-1234, can you check 
showtimes for Dune?',
            additional_kwargs={},
            response_metadata={},
            id='88dfbc60-ed9e-44eb-883b-423ebea90263'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 17,
                    'prompt_tokens': 225,
                    'total_tokens': 242,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': 0,
                        'cached_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 4.395e-05,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 4.395e-05,
                        'upstream_inference_prompt_cost': 3.375e-05,
                        'upstream_inference_completions_cost': 1.02e-05
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-4o-mini',
                'system_fingerprint': 'fp_f282d56213',
                'id': 'gen-1787428927-Eq26tqOQyP4oEOa5to2W',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a02b10-f794-7c71-90a2-c5c25d95e37c-0',
            tool_calls=[
                {
                    'name': 'check_showtime',
                    'args': {'movie_title': 'Dune'},
                    'id': 'call_qvnNA9a0Pz5oDkuZ3MC6ysiy',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 225,
                'output_tokens': 17,
                'total_tokens': 242,
                'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='No showtimes available for this movie.',
            name='check_showtime',
            id='2bf01e9c-da14-41db-8cf7-5cee51b96f83',
            tool_call_id='call_qvnNA9a0Pz5oDkuZ3MC6ysiy'
        ),
        AIMessage(
            content='There are currently no showtimes available for "Dune." If you have any other movies in mind or
need further assistance, feel free to ask!',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 31,
                    'prompt_tokens': 259,
                    'total_tokens': 290,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': 0,
                        'cached_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 5.745e-05,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 5.745e-05,
                        'upstream_inference_prompt_cost':

In [ ]:

print(result['messages'][-1].content)

In [38]:

import re
def detect_booking_code(content: str) -> list[dict]:
    """Detect CineBot's own booking code format: BK followed by 4 digits."""
    matches = []
    for match in re.finditer(r"BK\d{4}", content):
        matches.append({"text": match.group(0), "start": match.start(), "end": match.end()})
    return matches


custom_pii_agent = create_agent(
    model=model,
    tools=cinebot_tools,
    middleware=[PIIMiddleware("booking_code", detector=detect_booking_code, strategy="mask")],
)


result = custom_pii_agent.invoke({
    "messages": [("user", "Can you check the status of my booking BK1044 for me?")]
})

print(result)


{
    'messages': [
        HumanMessage(
            content='Can you check the status of my booking ****1044 for me?',
            additional_kwargs={},
            response_metadata={},
            id='fab8943f-72b8-4f9d-bb81-6434a37855ed'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 18,
                    'prompt_tokens': 202,
                    'total_tokens': 220,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': 0,
                        'cached_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 4.11e-05,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 4.11e-05,
                        'upstream_inference_prompt_cost': 3.03e-05,
                        'upstream_inference_completions_cost': 1.08e-05
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-4o-mini',
                'system_fingerprint': 'fp_f282d56213',
                'id': 'gen-1787428957-OtT3Y1fxZqt3M0VoTJvI',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a02b11-6c19-7d43-9bd1-711aea25ff9c-0',
            tool_calls=[
                {
                    'name': 'check_order_status',
                    'args': {'booking_id': '****1044'},
                    'id': 'call_FyjtvcG6K5GK2hwzzMSNZZaD',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 202,
                'output_tokens': 18,
                'total_tokens': 220,
                'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='Booking ****1044: confirmed, 2 seats, Interstellar, 7:00 PM.',
            name='check_order_status',
            id='13c8da42-9efd-432d-b7c2-3eac09386701',
            tool_call_id='call_FyjtvcG6K5GK2hwzzMSNZZaD'
        ),
        AIMessage(
            content='Your booking ****1044 is confirmed for 2 seats to see "Interstellar" at 7:00 PM.',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 25,
                    'prompt_tokens': 249,
                    'total_tokens': 274,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': 0,
                        'cached_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 5.235e-05,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 5.235e-05,
                        'upstream_inference_prompt_cost': 3.735e-05,
                        'upstream_inference_completions_cost': 1.5

#### TO Do List

In [39]:
todo_agent = create_agent(
    model=model,
    tools=cinebot_tools,
    middleware=[TodoListMiddleware()],
    system_prompt="You are a helpful assistant that can manage a user's to-do list of tasks."
)

In [40]:
result = todo_agent.invoke({
    "messages": [("user", "I want to plan a movie night: check what's showing, pick something good, and book 2 seats.")]
})

In [41]:
print(result)

{
    'messages': [
        HumanMessage(
            content="I want to plan a movie night: check what's showing, pick something good, and book 2 seats.",
            additional_kwargs={},
            response_metadata={},
            id='d2000775-815f-4000-9cc0-b29a396982c3'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 16,
                    'prompt_tokens': 1409,
                    'total_tokens': 1425,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': 0,
                        'cached_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 0.00022095,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 0.00022095,
                        'upstream_inference_prompt_cost': 0.00021135,
                        'upstream_inference_completions_cost': 9.6e-06
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-4o-mini',
                'system_fingerprint': 'fp_02ae59e84c',
                'id': 'gen-1787428978-0km7WmiB6KSNigjsFQ8s',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a02b11-c2cc-7660-a8ea-dc178ad67e28-0',
            tool_calls=[
                {
                    'name': 'check_showtime',
                    'args': {'movie_title': ''},
                    'id': 'call_9DK1lL06rYXTkbzUsliFtQg2',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 1409,
                'output_tokens': 16,
                'total_tokens': 1425,
                'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='No showtimes available for this movie.',
            name='check_showtime',
            id='5b62a98c-33b5-4fff-be1e-b9784c2fea04',
            tool_call_id='call_9DK1lL06rYXTkbzUsliFtQg2'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 18,
                    'prompt_tokens': 1442,
                    'total_tokens': 1460,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': 0,
                        'cached_tokens': 1408,
                        'video_tokens': 0
                    },
                    'cost': 0.0001215,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 0.0001215,
                        'upstream_inference_prompt_cost': 0.0001107,
                        'upstream_inference_completions_cost': 1.08e-05
                    }
                },
                'mode

#### LLM Tool Selector

In [42]:
from langchain.agents.middleware import wrap_model_call

@wrap_model_call
def show_tools(request, handler):
    print("\nTOOLS SENT TO MODEL:")
    print([tool.name for tool in request.tools])

    return handler(request)

In [43]:
selector_agent = create_agent(
    model=model,
    tools=cinebot_tools,
    middleware=[
        LLMToolSelectorMiddleware(
            model=free_selector_model,    # can be a CHEAPER model than the main agent
            max_tools=2,
            always_include=["check_showtime"],  # always kept, doesn't count against max_tools
        ),
        show_tools
    ],
)

result = selector_agent.invoke({"messages": [("user", "Can you cancel my booking with ID B1234?")]})

TOOLS SENT TO MODEL:

['cancel_booking', 'check_showtime']

OutputParserException: Invalid json output: User Safety: safe
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 

#### Tool Error

In [44]:
def handle_tool_error(exc: Exception, request) -> str | None:
    tool_name = request.tool_call["name"]

    if isinstance(exc, ValueError):
        return (
            f"Tool '{tool_name}' failed because of invalid input. "
            "Please correct the arguments and try again."
        )

    # Returning None allows unexpected errors to propagate normally.
    return None


tool_error_agent = create_agent(
    model=model,
    tools=cinebot_tools,
    middleware=[
        ToolErrorMiddleware(
            on_error=handle_tool_error,
            tools=["lookup_seat_map"],
        ),
    ],
)

In [45]:
result = tool_error_agent.invoke({
    "messages": [
        (
            "user",
            "Check seat 123 for Interstellar."
        )
    ]
})

print(result)

{
    'messages': [
        HumanMessage(
            content='Check seat 123 for Interstellar.',
            additional_kwargs={},
            response_metadata={},
            id='0ff30c7c-5b69-4965-a6fb-f7e2379ed903'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 23,
                    'prompt_tokens': 196,
                    'total_tokens': 219,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': 0,
                        'cached_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 4.32e-05,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 4.32e-05,
                        'upstream_inference_prompt_cost': 2.94e-05,
                        'upstream_inference_completions_cost': 1.38e-05
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-4o-mini',
                'system_fingerprint': 'fp_f282d56213',
                'id': 'gen-1787429124-NQvxIQ1QiDEEWm6aClCx',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a02b13-faa6-7292-8a3d-d6e72cc46913-0',
            tool_calls=[
                {
                    'name': 'lookup_seat_map',
                    'args': {'movie_title': 'Interstellar', 'seat_number': '123'},
                    'id': 'call_QtvzUdz25lSImzme5kqLBEzo',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 196,
                'output_tokens': 23,
                'total_tokens': 219,
                'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content="Tool 'lookup_seat_map' failed because of invalid input. Please correct the arguments and try 
again.",
            name='lookup_seat_map',
            id='33c47e1f-58b0-44c6-9788-f983e4f96be7',
            tool_call_id='call_QtvzUdz25lSImzme5kqLBEzo',
            status='error'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 24,
                    'prompt_tokens': 250,
                    'total_tokens': 274,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': 0,
                        'cached_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 5.19e-05,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 5.19e-05,
                        'upstream_inference_prompt_cost': 3.75e-05,
                        'upstream_inference_completions_cost': 1.44e-05
           

#### Tool retry

In [ ]:
tool_retry_agent = create_agent(
    model=model,
    tools=cinebot_tools,
    middleware=[
        ToolRetryMiddleware(
            max_retries=2,
            tools=["lookup_seat_map"],
            retry_on=(ValueError,),
            on_failure="continue",
            backoff_factor=0.0,
            initial_delay=0.5,
        ),
    ],
)

In [ ]:
result = tool_retry_agent.invoke({
    "messages": [
        ("user", "Check seat 123 for Interstellar.")
    ]
})

print(result)

In [ ]:
def format_tool_failure(exc: Exception) -> str:
    return (
        "The tool failed after multiple attempts. "
        "Please provide valid input or try again later."
    )


tool_retry_agent = create_agent(
    model=model,
    tools=cinebot_tools,
    middleware=[
        ToolRetryMiddleware(
            max_retries=2,
            tools=["lookup_seat_map"],
            retry_on=(ValueError,),
            on_failure=format_tool_failure,
            backoff_factor=0.0,
            initial_delay=0.5,
        ),
    ],
)

result = tool_retry_agent.invoke({
    "messages": [
        ("user", "Check seat 123 for Interstellar.")
    ]
})

print(result)

#### LLM tool emulator

In [46]:
emulated_tool_agent = create_agent(
    model=model,
    tools=cinebot_tools,
    middleware=[
        LLMToolEmulator(
            tools=["lookup_seat_map"],
            model = model
        )
    ],
)



In [47]:
result = emulated_tool_agent.invoke({
    "messages": [
        ("user", "Check seat A12 for Interstellar.")
    ]
})

print(result)

{
    'messages': [
        HumanMessage(
            content='Check seat A12 for Interstellar.',
            additional_kwargs={},
            response_metadata={},
            id='7f63e463-60a0-4c6f-b756-0f93073fea39'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 24,
                    'prompt_tokens': 196,
                    'total_tokens': 220,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': 0,
                        'cached_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 4.38e-05,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 4.38e-05,
                        'upstream_inference_prompt_cost': 2.94e-05,
                        'upstream_inference_completions_cost': 1.44e-05
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-4o-mini',
                'system_fingerprint': 'fp_f282d56213',
                'id': 'gen-1787429198-b3HH27HSyRXv9rII45ek',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a02b15-19d2-7033-b629-f1a9d4b1cb11-0',
            tool_calls=[
                {
                    'name': 'lookup_seat_map',
                    'args': {'movie_title': 'Interstellar', 'seat_number': 'A12'},
                    'id': 'call_cdPXc0uJg00Vw0fw3LZPQREd',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 196,
                'output_tokens': 24,
                'total_tokens': 220,
                'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='{"status": "success", "movie_title": "Interstellar", "seat_number": "A12", "availability": 
"available", "row": "A", "seat": 12}',
            name='lookup_seat_map',
            id='b9ed1ae4-b5f5-4b88-a8ef-9c6d4dcd9d33',
            tool_call_id='call_cdPXc0uJg00Vw0fw3LZPQREd'
        ),
        AIMessage(
            content='Seat A12 for "Interstellar" is available.',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 12,
                    'prompt_tokens': 270,
                    'total_tokens': 282,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': 0,
                        'cached_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 4.77e-05,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 4.77e-05,
                        'upstream_inference_prompt_cost': 4.05e-05,
                        'upstream_inferenc